Sorry, wrong data folder. should be AmonZ instead of Amon
So this code is not used

In [ ]:
import xarray as xr
import numpy as np

# File paths (same as before)
ua_file1 = "/mnt/backup_ETH/MIROC-ES2H/PD-INT/r1i1p1f1/Amon/ua/gn/v20240220/ua_Amon_MIROC-ES2H_PD-INT_r1i1p1f1_gn_202001-206912.nc"
ua_file2 = "/mnt/backup_ETH/MIROC-ES2H/PD-INT/r1i1p1f1/Amon/ua/gn/v20240517/ua_Amon_MIROC-ES2H_PD-INT_r1i1p1f1_gn_207001-207412.nc"

# Load datasets
ua_ds1 = xr.open_dataset(ua_file1)
ua_ds2 = xr.open_dataset(ua_file2)
ps = xr.concat([ua_ds1['ps'], ua_ds2['ps']], dim='time')

# Get fill value (default assumed to be 9.96921e+36, or use attribute value if available)
fill_value = ps.attrs.get('_FillValue', 9.96921e+36)

# Calculate the number and percentage of missing values
total_elements = ps.size  # Total number of data points (time, lat, lon)
missing_elements = np.sum(ps == fill_value)  # Number of fill values
missing_ratio = (missing_elements / total_elements) * 100  # Percentage of missing values (%)

# Check the number and percentage of NaN values (if any NaNs exist in the data)
nan_elements = np.sum(np.isnan(ps))
nan_ratio = (nan_elements / total_elements) * 100

# Print results
print(f"Total data points: {total_elements}")
print(f"Number of fill values ({fill_value}): {missing_elements}")
print(f"Fill value percentage: {missing_ratio:.2f}%")
print(f"Number of NaN values: {nan_elements}")
print(f"NaN value percentage: {nan_ratio:.2f}%")
print(f"Total missing value percentage (fill values + NaN): {(missing_elements + nan_elements) / total_elements * 100:.2f}%")

# Print ps attributes
print("ps attributes:", ps.attrs)

# Print first few values
print("First 5 values of ps (time=0, lat=0, lon=0-4):", ps.isel(time=0, lat=0, lon=slice(0, 5)).values)

# Check if all values are fill values
fill_value = ps.attrs.get('_FillValue', 9.96921e+36)
unique_values = np.unique(ps.values)
print("Unique values of ps:", unique_values)

# Close datasets
ua_ds1.close()
ua_ds2.close()


In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy import signal

# Define output directory for saving plots
out_dir = "/home/weiji/restart_exam/20250307/MIROC/attempt"
os.makedirs(out_dir, exist_ok=True)  # Create output directory if it doesn't exist

# Define input file paths
o3_file1 = "/mnt/backup_ETH/MIROC-ES2H/PD-INT/r1i1p1f1/Amon/o3/gn/v20240220/o3_Amon_MIROC-ES2H_PD-INT_r1i1p1f1_gn_202001-206912.nc"
o3_file2 = "/mnt/backup_ETH/MIROC-ES2H/PD-INT/r1i1p1f1/Amon/o3/gn/v20240517/o3_Amon_MIROC-ES2H_PD-INT_r1i1p1f1_gn_207001-207412.nc"
ua_file1 = "/mnt/backup_ETH/MIROC-ES2H/PD-INT/r1i1p1f1/Amon/ua/gn/v20240220/ua_Amon_MIROC-ES2H_PD-INT_r1i1p1f1_gn_202001-206912.nc"
ua_file2 = "/mnt/backup_ETH/MIROC-ES2H/PD-INT/r1i1p1f1/Amon/ua/gn/v20240517/ua_Amon_MIROC-ES2H_PD-INT_r1i1p1f1_gn_207001-207412.nc"
ta_file1 = "/mnt/backup_ETH/MIROC-ES2H/PD-INT/r1i1p1f1/Amon/ta/gn/v20240220/ta_Amon_MIROC-ES2H_PD-INT_r1i1p1f1_gn_202001-206912.nc"
ta_file2 = "/mnt/backup_ETH/MIROC-ES2H/PD-INT/r1i1p1f1/Amon/ta/gn/v20240517/ta_Amon_MIROC-ES2H_PD-INT_r1i1p1f1_gn_207001-207412.nc"

# Read and concatenate O3 data
o3_ds1 = xr.open_dataset(o3_file1)
o3_ds2 = xr.open_dataset(o3_file2)
o3 = xr.concat([o3_ds1['o3'], o3_ds2['o3']], dim='time')

# Read and concatenate U data
ua_ds1 = xr.open_dataset(ua_file1)
ua_ds2 = xr.open_dataset(ua_file2)
ua = xr.concat([ua_ds1['ua'], ua_ds2['ua']], dim='time')

# Read and concatenate T data
ta_ds1 = xr.open_dataset(ta_file1)
ta_ds2 = xr.open_dataset(ta_file2)
ta = xr.concat([ta_ds1['ta'], ta_ds2['ta']], dim='time')

# Get coordinate variables (using the first O3 file as reference)
time = xr.concat([o3_ds1['time'], o3_ds2['time']], dim='time')
lev = ua.lev  # Use ua dataset for lev since we're plotting U
lat = ua.lat
lon = ua.lon

# Close datasets to free memory
o3_ds1.close()
o3_ds2.close()
ua_ds1.close()
ua_ds2.close()
ta_ds1.close()
ta_ds2.close()


# Verify if data is monthly mean by checking time intervals
time_diff = ua.time.diff('time').values

# Convert time to decimal years for X-axis
years = ua.time.dt.year + ua.time.dt.month / 12.0



# Select equatorial data (latitudes between -5° and 5°, averaged over lat and lon)
ua_equator = ua.sel(lat=slice(-5, 5)).mean(dim='lat').mean(dim='lon')

# Deseasonalize: Remove monthly climatology
monthly_clim = ua_equator.groupby('time.month').mean('time')
ua_deseasoned = ua_equator.groupby('time.month') - monthly_clim

# Convert hybrid sigma-pressure levels to height (km)
# Since lev is hybrid sigma-pressure (p = a*p0 + b*ps), we approximate height using a log-pressure assumption
# Standard atmosphere: H = -H_scale * ln(p/p0), H_scale ~ 7 km, p0 = 1000 hPa
# For simplicity, assume typical pressure levels and convert to height
p0_value = 1000  # Reference pressure in hPa
H_scale = 7.0  # Scale height in km
lev_values = lev.values  # Hybrid sigma-pressure coordinate
pressure = lev_values * p0_value  # Approximate pressure (simplification)
height = -H_scale * np.log(pressure / p0_value)  # Formula: H = -H_scale * ln(p/p0)

# Limit height to 18–60 km
height_mask = (height >= 18) & (height <= 60)
height_limited = height[height_mask]
ua_deseasoned_limited = ua_deseasoned.isel(lev=height_mask)

In [ ]:
# Create vertical profile plot
plt.figure(figsize=(20, 4))
plt.contourf(years, height_limited, ua_deseasoned_limited.T, cmap='RdBu_r', levels=20)

plt.colorbar(label='Deseasoned Zonal Wind (m/s)')
plt.xlabel('Year')
plt.ylabel('Height (km)')
plt.title('Equatorial Zonal Wind Vertical Profile (Deseasoned Monthly Mean)')
plt.ylim(18, 60)  # Explicitly set Y-axis limits
plt.xlim(2030, 2060)  # Explicitly set Y-axis limits
# Save the plot
out_plot = os.path.join(out_dir, "equatorial_zonal_wind_profile.png")
plt.savefig(out_plot, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Option 1: If you have a way to convert to pressure (hPa)
# If your dataset has surface pressure (ps) or a hybrid-to-pressure conversion
# You'll need to adjust this based on your actual data structure
try:
    # Assuming you might have surface pressure or a conversion method
    # This is a simplified example - replace with your actual conversion
    # For hybrid coords: p = a + b*ps, where ps is surface pressure
    if 'ps' in ua_deseasoned.coords:  # If surface pressure exists
        ps = ua_deseasoned['ps'].mean()  # Average surface pressure
        # Simplified conversion (you'd need actual a and b coefficients)
        pressure = lev * ps  # This is approximate!
    else:
        # Fallback: assume linear mapping from hybrid to pressure
        pressure = lev * 1000  # Rough scale from 1000 hPa to 0
        
    # Select 100 hPa to 1 hPa range
    mask = (pressure <= 100) & (pressure >= 1)
    lev_limited = pressure[mask]
    ua_limited = ua_deseasoned.isel(lev=mask)
except:
    # Option 2: If no conversion available, use mid-levels and note limitation
    print("Warning: Using hybrid coordinates as proxy. For accurate hPa, provide conversion.")
    mid_index = len(lev) // 2
    lev_limited = lev.isel(lev=slice(0, mid_index))
    ua_limited = ua_deseasoned.isel(lev=slice(0, mid_index))

# Verify shapes
print("After limiting:")
print("lev_limited shape:", lev_limited.shape)
print("ua_limited shape:", ua_limited.shape)

# Create plot
plt.figure(figsize=(24, 4))
contour = plt.contourf(years, lev_limited, ua_limited.T, levels=20, cmap='RdBu_r')
plt.contour(years, lev_limited, ua_limited.T, levels=[0], colors='black', linewidths=1)
plt.colorbar(contour, label='Deseasonalized U Wind (m/s)')

# Set axes
plt.xlabel('Year')
plt.ylabel('Pressure Level (hPa)')
plt.yscale('log')  # Logarithmic scale for Y-axis
plt.gca().invert_yaxis()  # Pressure decreases upward

# Adjust Y-axis ticks for better readability
plt.gca().set_yticks([1, 5, 10, 20, 50, 100])
plt.gca().set_ylim(100, 1)  # Set explicit limits

# Add title and grid
plt.title('Equatorial Zonal Wind Anomalies (100 hPa to 1 hPa)')
plt.grid(True, alpha=0.3, which='both')

# Save figure
plt.savefig(os.path.join(out_dir, 'equatorial_zonal_wind_pressure_levels_log.png'), 
            dpi=300, bbox_inches='tight')
plt.show()